In [1]:
!pip install xgboost scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

print("Imports successful")

Imports successful


In [2]:
data         = joblib.load("/kaggle/input/datasets/pes2ug23cs197/splits/splits.pkl")
X_train      = data["X_train"]
X_val        = data["X_val"]
X_test       = data["X_test"]
y_train      = data["y_train"]
y_val        = data["y_val"]
y_test       = data["y_test"]
del data
gc.collect()

le           = joblib.load("/kaggle/input/datasets/pes2ug23cs197/splits/label_encoder.pkl")
feature_cols = joblib.load("/kaggle/input/datasets/pes2ug23cs197/splits/feature_cols.pkl")
num_classes  = len(le.classes_)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Classes: {num_classes} | Features: {X_train.shape[1]}")

Train: 34999 | Val: 7500 | Test: 7500
Classes: 1117 | Features: 1756


In [3]:
class_weights_arr = compute_class_weight(
    class_weight = "balanced",
    classes      = np.unique(y_train),
    y            = y_train
)

class_weight_dict = dict(zip(np.unique(y_train), class_weights_arr))
sample_weights    = np.array([class_weight_dict[c] for c in y_train], dtype=np.float32)
print("Sample weights computed")
print(f"Weight range: {sample_weights.min():.4f} – {sample_weights.max():.4f}")

Sample weights computed
Weight range: 0.3164 – 12.0770


In [4]:
from sklearn.preprocessing import LabelEncoder

le_xgb      = LabelEncoder()
y_train_enc = le_xgb.fit_transform(y_train)
y_val_enc   = le_xgb.transform(y_val)
y_test_enc  = le_xgb.transform(y_test)

print(f"Original labels range: {y_train.min()} – {y_train.max()}")
print(f"Encoded labels range:  {y_train_enc.min()} – {y_train_enc.max()}")
print(f"Num classes: {len(le_xgb.classes_)}")

Original labels range: 3 – 1115
Encoded labels range:  0 – 482
Num classes: 483


In [5]:
xgb_model = XGBClassifier(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 5,          
    min_child_weight      = 5,          
    subsample             = 0.75,       
    colsample_bytree      = 0.65,       
    colsample_bylevel     = 0.65,       
    reg_alpha             = 0.08,       
    reg_lambda            = 1.5,        
    gamma                 = 0.2,        
    tree_method           = "hist",
    device                = "cpu",
    objective             = "multi:softprob",
    eval_metric           = "mlogloss",
    early_stopping_rounds = 30,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 1
)

print("Training XGBoost ...")
xgb_model.fit(
    X_train, y_train_enc,
    sample_weight = sample_weights,
    eval_set      = [(X_val, y_val_enc)],
    verbose       = 50
)

print(f"Best iteration: {xgb_model.best_iteration}")
joblib.dump(xgb_model, "xgb_model.pkl")
joblib.dump(le_xgb,    "le_xgb.pkl")
print("xgb_model.pkl and le_xgb.pkl saved")

Training XGBoost ...
[0]	validation_0-mlogloss:6.11459
[50]	validation_0-mlogloss:3.50862
[100]	validation_0-mlogloss:1.79455
[150]	validation_0-mlogloss:1.19122
[200]	validation_0-mlogloss:0.95662
[250]	validation_0-mlogloss:0.83439
[300]	validation_0-mlogloss:0.75738
[350]	validation_0-mlogloss:0.71114
[400]	validation_0-mlogloss:0.67957
[450]	validation_0-mlogloss:0.65454
[499]	validation_0-mlogloss:0.63580
Best iteration: 499
xgb_model.pkl and le_xgb.pkl saved


In [6]:
y_val_pred   = le_xgb.inverse_transform(xgb_model.predict(X_val))
y_train_pred = le_xgb.inverse_transform(xgb_model.predict(X_train))

val_acc   = accuracy_score(y_val,   y_val_pred)
train_acc = accuracy_score(y_train, y_train_pred)
gap       = train_acc - val_acc

print(f"Train Accuracy : {train_acc*100:.2f}%")
print(f"Val   Accuracy : {val_acc*100:.2f}%")
print(f"Gap            : {gap*100:.2f}%  {'- OK' if gap < 0.10 else '- Check regularization'}")

Train Accuracy : 88.49%
Val   Accuracy : 81.45%
Gap            : 7.03%  - OK


In [7]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  

xgb_cv = XGBClassifier(
    n_estimators      = 500,           
    learning_rate     = 0.05,
    max_depth         = 5,             
    min_child_weight  = 5,             
    subsample         = 0.75,          
    colsample_bytree  = 0.65,          
    colsample_bylevel = 0.65,          
    reg_alpha         = 0.08,          
    reg_lambda        = 1.5,           
    gamma             = 0.2,           
    tree_method       = "hist",
    device            = "cpu",
    objective         = "multi:softprob",
    eval_metric       = "mlogloss",
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0
)

cv_scores = cross_val_score(
    xgb_cv, X_train, y_train_enc,     
    cv      = skf,
    scoring = "accuracy",
    n_jobs  = 1
)

print(f"CV Scores : {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean      : {cv_scores.mean()*100:.2f}%")
print(f"Std Dev   : {cv_scores.std()*100:.2f}%  {'- Stable' if cv_scores.std() < 0.02 else '- High variance'}")

CV Scores : ['0.6746', '0.6626', '0.6696']
Mean      : 66.89%
Std Dev   : 0.49%  - Stable
